# Uttarakhand Wildfire Feature Stack — starter

Weather, terrain, land cover and MODIS active fire, co-registered on one
~1 km grid at hourly resolution. 311 x 400 cells, 1,464 hours (Apr-May 2016).

This notebook covers the four things that are easy to get wrong with this file:

1. opening it (the netCDF engine matters),
2. **which of the three fire channels to use**,
3. **the persistence baseline you must beat** — computed live below, not quoted,
4. loading it without exhausting RAM.

In [ ]:
import os
# HDF5 takes locks that some filesystems do not support. Harmless on Kaggle,
# essential on FUSE mounts (ntfs-3g, sshfs). Must be set BEFORE the import.
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

ROOT = "/kaggle/input/uttarakhand-wildfire-dataset"
NC   = f"{ROOT}/final_feature_stack_RELEASE_v2.nc"

# Let xarray pick the engine. Kaggle ships h5netcdf but NOT netcdf4, so
# pinning engine="netcdf4" raises "unrecognized engine" here. Both read this
# file identically. (Only if you also WRITE in the same process must you avoid
# mixing the two -- they bundle separate HDF5 libraries.)
ds = xr.open_dataset(NC)   # lazy: nothing is read yet
ds

## Do not call `.values` on the whole thing

The file is 2.7 GiB on disk, but that is *compressed*. Decompressed, the seven
hourly float32 weather fields are about 5 GiB, and `ds.to_array()` would
materialise all of it at once.

`open_dataset` is lazy. Slice first, then read.

In [ ]:
n_t, n_y, n_x = ds.sizes["valid_time"], ds.sizes["latitude"], ds.sizes["longitude"]
hourly = [v for v in ds.data_vars if "valid_time" in ds[v].dims]
static = [v for v in ds.data_vars if "valid_time" not in ds[v].dims]

print(f"grid   : {n_t} hours x {n_y} lat x {n_x} lon")
print(f"hourly : {hourly}")
print(f"static : {static}")

gib = lambda n: n * n_t * n_y * n_x * 4 / 1024**3
print(f"\nall {len(hourly)} hourly fields as float32 = {gib(len(hourly)):.1f} GiB in RAM")
print("-> slice a time window, or load one variable at a time.")

# The three fire channels are int8, so they are cheap enough to hold whole.
print(f"one int8 fire channel          = {n_t*n_y*n_x/1024**2:.0f} MiB")

## The three fire channels answer different questions

Pick deliberately. This is the single most important choice you make here.

| Channel | What it is | Use it for |
|---|---|---|
| `OBSERVED_FIRE` | exactly what MODIS saw, at the hour it saw it | evaluation against ground truth |
| `ACTIVE_FIRE` | detections held alight 12 h, forward in time only | **the intended prediction target** |
| `BURNED_AREA` | running maximum of `ACTIVE_FIRE` | context feature — never a target |

`BURNED_AREA` is monotone by construction: it can only grow. A model that
predicts it at t+1 learns "output whatever was on at t, plus a bit", which
scores extremely well and means nothing.

And note **`OBSERVED_FIRE[t]` determines `ACTIVE_FIRE[t..t+12]` by
construction** — so if you train on `ACTIVE_FIRE`, `OBSERVED_FIRE` must not be
in your inputs. That is a label leak, not a feature.

In [ ]:
fire = {v: ds[v].values.astype(bool) for v in ("OBSERVED_FIRE", "ACTIVE_FIRE", "BURNED_AREA")}

for name, arr in fire.items():
    print(f"{name:14s} {arr.sum():>10,} pixel-hours   "
          f"positive rate {100*arr.mean():.4f}%")

obs = fire["OBSERVED_FIRE"]
frames_with_new_fire = int((obs.sum(axis=(1, 2)) > 0).sum())
print(f"\nhours containing a new detection: {frames_with_new_fire} of {n_t}")
print("MODIS passes ~4x/day, so most hours carry no new information.")

## The persistence baseline — run this before you report a score

The dumbest possible model copies its input frame to its output. Here is what
that scores. Any IoU at or below the figure for your lead time is **not
evidence of skill**.

In [ ]:
def iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union else np.nan

act = fire["ACTIVE_FIRE"]
rows = [(h, iou(act[:-h], act[h:])) for h in (1, 4, 8, 12, 24, 48)]
base = pd.DataFrame(rows, columns=["lead_hours", "persistence_IoU"])
print(base.to_string(index=False, float_format="%.4f"))

Read that table carefully:

- **1 h** — 0.85. The task is nearly trivial; almost nothing changes in an hour.
- **8 h** — the most informative setting. Inside the 12 h persistence window,
  but far enough out that copying is clearly not enough.
- **24 h and beyond** — the persistence window has fully elapsed, so the task
  becomes predicting *new ignitions*. Expect a low absolute IoU and judge it
  against ~0.01, not against 1.0.

If you report "IoU 0.84 at 1 h lead", you have reported the baseline.

In [ ]:
plt.figure(figsize=(6, 3.5))
plt.plot(base.lead_hours, base.persistence_IoU, "o-")
plt.axvline(12, ls="--", c="grey", lw=1)
plt.text(12.4, 0.55, "12 h persistence\nwindow ends", fontsize=8, color="grey")
plt.xlabel("lead time (hours)"); plt.ylabel("persistence IoU")
plt.title("Copy-the-input baseline on ACTIVE_FIRE"); plt.yscale("log")
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## What it looks like

In [ ]:
# Busiest hour, for a frame that actually shows something.
t = int(act.sum(axis=(1, 2)).argmax())
stamp = pd.to_datetime(ds.valid_time.values[t])

dem  = ds["DEM"].values
lulc = ds["LULC"].values

fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
ext = [float(ds.longitude.min()), float(ds.longitude.max()),
       float(ds.latitude.min()),  float(ds.latitude.max())]

# DEM == 0 is a NODATA sentinel, not sea level. Mask it or the colour
# scale collapses and the terrain reads as a flat plain.
m = axes[0].imshow(np.where(dem > 0, dem, np.nan), extent=ext, cmap="terrain")
axes[0].set_title("DEM (0 masked as nodata)"); plt.colorbar(m, ax=axes[0], label="m")

axes[1].imshow(lulc, extent=ext, cmap="tab20", interpolation="nearest")
axes[1].set_title("LULC class codes")

axes[2].imshow(np.where(dem > 0, dem, np.nan), extent=ext, cmap="Greys_r", alpha=.6)
yy, xx = np.nonzero(act[t])
axes[2].scatter(ds.longitude.values[xx], ds.latitude.values[yy], s=6, c="red")
axes[2].set_title(f"ACTIVE_FIRE @ {stamp:%Y-%m-%d %H:%M} UTC")

for a in axes: a.set_xlabel("lon"); a.set_ylabel("lat")
plt.tight_layout(); plt.show()

## Land cover: ESA WorldCover, 10 m aggregated to 1 km

Real class codes with real names, CC-BY-4.0, and full coverage of the grid —
there is no off-map background and no unclassified fill.

Each 1 km cell holds the **areal majority** class over ~11,664 source pixels.
That is lossy in one specific way worth knowing: mean purity of the winning
class is 0.755, and in 8.3% of cells the winner holds under half the cell, so
the "majority" is really a plurality. Forest/cropland boundaries are the usual
case.

`is_burnable` is our judgement, not ESA's: it flags classes that can carry a
vegetation fire. Use it as a loss mask — asking a model to predict fire on
permanent water or snow is free, misleading accuracy.

In [ ]:
legend = pd.read_csv(f"{ROOT}/worldcover_legend.csv")
display(legend)

BURNABLE = tuple(legend.loc[legend.is_burnable, "code"])
burnable = np.isin(lulc, BURNABLE)
print(f"burnable classes    : {BURNABLE}")
print(f"grid that is burnable: {100*burnable.mean():.1f}%")
print(f"unclassified cells   : {(lulc == 0).sum()}")

ever = fire["BURNED_AREA"][-1]
print(f"burned cells on burnable land: {100*burnable[ever].mean():.1f}%")

names = dict(zip(legend.code, legend.name))
for c, n in sorted(zip(*np.unique(lulc[ever], return_counts=True)), key=lambda t: -t[1]):
    print(f"   {names[c]:<26} {n:>5} burned cells")

print("\n-> use `burnable` as a loss mask; do not score on water, snow or bare rock.")

## Splitting it — the part that bites

`ACTIVE_FIRE` is 0.017% positive, and only ~125 hours introduce new fire. A
naive chronological 80/20 split leaves you roughly **nine** validation events,
which is far too few for the number to mean anything.

A naive north/south spatial split is worse: fire is not uniform, so one half
can end up with almost no fire at all.

In [ ]:
split = int(0.8 * n_t)
new_fire_hours = np.nonzero(obs.sum(axis=(1, 2)) > 0)[0]
print(f"chronological 80/20 -> {(new_fire_hours >= split).sum()} validation events. Too few.")

# Better: block the timeline into days and stratify the blocks by fire activity,
# so train and validation each see a comparable mix of quiet and active days.
day = np.arange(n_t) // 24
per_day = pd.Series(obs.sum(axis=(1, 2))).groupby(day).sum()
active_days = per_day[per_day > 0].index.to_numpy()

rng = np.random.default_rng(0)
val_days = set(rng.choice(active_days, size=max(1, len(active_days) // 5), replace=False))
val_mask = np.isin(day, list(val_days))

print(f"stratified day-block -> {int((obs[val_mask].sum(axis=(1,2))>0).sum())} validation events")
print(f"   train hours {(~val_mask).sum()}, val hours {val_mask.sum()}")
print("\nWhole DAYS, never individual hours: neighbouring hours are near-identical,")
print("so an hour-level shuffle puts a near-copy of every val frame in train.")

## Loading a training window

Slice the time axis first. This reads only the hours you asked for.

In [ ]:
FEATURES = ["t2m", "d2m", "u10", "v10", "swvl1", "tp", "e"]   # hourly
STATICS  = ["DEM", "LULC", "GHS_BUILT", "cvl"]                # broadcast over time

def load_window(t0, n_hours=24):
    w = ds.isel(valid_time=slice(t0, t0 + n_hours))
    x = np.stack([w[v].values for v in FEATURES], axis=1)        # (T, C, y, x)
    s = np.stack([ds[v].values.astype("float32") for v in STATICS])
    s = np.broadcast_to(s, (n_hours, *s.shape))
    x = np.concatenate([x, s], axis=1)
    y = w["ACTIVE_FIRE"].values[:, None].astype("float32")       # (T, 1, y, x)
    return x, y

x, y = load_window(int(new_fire_hours[len(new_fire_hours)//2]) - 12)
print(f"x {x.shape}  y {y.shape}")
print(f"channels: {FEATURES + STATICS}")
print(f"\nNote OBSERVED_FIRE and BURNED_AREA are absent from the inputs -- both")
print("leak the ACTIVE_FIRE target. Add BURNED_AREA back only if you shift it.")

## Before you report a result

- Did you beat the persistence IoU **for your lead time**? (table above)
- Is `OBSERVED_FIRE` out of your inputs?
- Are you scoring only on `burnable` cells?
- Are you masking `DEM == 0`?
- Did you split by day-blocks rather than by hour?

Full provenance, per-variable units and the remaining caveats are in the
dataset description.